# CRAWLING DETIK.COM

## Instalasi Library

Kode ini digunakan untuk menginstal beberapa library Python yang dibutuhkan untuk melakukan web scraping dan pengolahan data, yaitu requests, beautifulsoup4, trafilatura, pandas, openpyxl, dan tqdm.  

In [1]:
# Jika belum install library, jalankan sekali saja
!pip install requests beautifulsoup4 trafilatura pandas openpyxl tqdm


   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -----

## Import Library dan Konfigurasi Awal

Bagian ini mengimpor semua modul Python yang diperlukan untuk scraping (seperti re, requests, BeautifulSoup, trafilatura, dll). Selain itu, kode ini melakukan konfigurasi HTTP Header (User-Agent) agar permintaan web tidak diblokir oleh server, serta mendefinisikan URL target kategori berita (sport dan finance) dari situs web Detik

In [2]:
import re
import time
import random
import requests
import pandas as pd
import trafilatura

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlsplit, urlunsplit
from tqdm import tqdm


# ============================================================
# 1. KONFIGURASI
# ============================================================

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/152.0 Safari/537.36"
    )
}

session = requests.Session()
session.headers.update(HEADERS)


kategori = {
    "sport": {
        "url": "https://sport.detik.com/indeks",
        "domain": "sport.detik.com"
    },

    "finance": {
        "url": "https://finance.detik.com/indeks",
        "domain": "finance.detik.com"
    }
}

## Fungsi Normalisasi URL

Kode ini membuat sebuah fungsi bernama normalize_url yang bertugas untuk membersihkan URL. Fungsi ini menghapus parameter query dan fragment dari tautan web untuk mencegah pengambilan URL yang duplikat nantinya.

In [3]:
def normalize_url(url):
    """
    Menghapus query parameter dan fragment dari URL.
    """

    bagian = urlsplit(url)

    url_bersih = urlunsplit(
        (
            bagian.scheme,
            bagian.netloc,
            bagian.path,
            "",
            ""
        )
    )

    return url_bersih

## Fungsi Pengambilan Tautan Berita

Fungsi ambil_link_berita dirancang untuk menelusuri halaman indeks berita (berdasarkan pagination) dan mengekstrak semua tautan artikel. Fungsi ini menyaring URL agar hanya mengambil tautan artikel dengan format spesifik milik Detik (memiliki pola /d-angka) dan mengumpulkan tautan tersebut hingga mencapai target jumlah yang ditentukan (misalnya 150 tautan).

In [4]:
def ambil_link_berita(index_url, domain, target=150, max_page=20):

    links = []
    sudah_ada = set()

    for page in range(1, max_page + 1):

        url_page = f"{index_url}?page={page}"

        print(f"Membaca halaman {page}")

        try:

            response = session.get(
                url_page,
                timeout=20
            )

            response.raise_for_status()

        except Exception as e:

            print("Error:", e)
            continue


        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )


        for tag in soup.find_all("a", href=True):

            href = urljoin(
                url_page,
                tag["href"]
            )

            href = normalize_url(href)

            parsed = urlsplit(href)


            # URL artikel Detik biasanya memiliki /d-xxxx
            if (
                parsed.netloc == domain
                and re.search(r"/d-\d+", parsed.path)
            ):

                if href not in sudah_ada:

                    sudah_ada.add(href)

                    links.append(href)


        print(
            "Jumlah link:",
            len(links)
        )


        if len(links) >= target:
            break


        # delay
        time.sleep(
            random.uniform(1, 2)
        )


    return links

## Scraping Link Kategori Sport

Kode ini memanggil fungsi ambil_link_berita untuk mencari dan mengumpulkan hingga 150 tautan artikel khusus dari kategori "sport".

In [5]:
link_sport = ambil_link_berita(
    kategori["sport"]["url"],
    kategori["sport"]["domain"],
    target=150
)

print(
    "Total kandidat Sport:",
    len(link_sport)
)

Membaca halaman 1
Jumlah link: 20
Membaca halaman 2
Jumlah link: 40
Membaca halaman 3
Jumlah link: 60
Membaca halaman 4
Jumlah link: 79
Membaca halaman 5
Jumlah link: 98
Membaca halaman 6
Jumlah link: 117
Membaca halaman 7
Jumlah link: 137
Membaca halaman 8
Jumlah link: 157
Total kandidat Sport: 157


## Scraping Link Kategori Finance

Kode ini memanggil fungsi ambil_link_berita kembali, namun kali ini untuk mencari dan mengumpulkan hingga 150 tautan artikel khusus dari kategori "finance"

In [6]:
link_finance = ambil_link_berita(
    kategori["finance"]["url"],
    kategori["finance"]["domain"],
    target=150
)

print(
    "Total kandidat Finance:",
    len(link_finance)
)

Membaca halaman 1
Jumlah link: 20
Membaca halaman 2
Jumlah link: 38
Membaca halaman 3
Jumlah link: 58
Membaca halaman 4
Jumlah link: 77
Membaca halaman 5
Jumlah link: 97
Membaca halaman 6
Jumlah link: 114
Membaca halaman 7
Jumlah link: 134
Membaca halaman 8
Jumlah link: 154
Total kandidat Finance: 154


## Fungsi Pembersihan Teks Berita

Fungsi bersihkan_teks dibuat untuk merapikan teks artikel yang telah diekstrak. Fungsi ini secara otomatis menghapus kata-kata tidak penting seperti "ADVERTISEMENT" dan "SCROLL TO CONTINUE WITH CONTENT", serta merapikan spasi yang berlebihan di dalam teks.

In [7]:
def bersihkan_teks(teks):

    if teks is None:
        return None


    # menghapus teks iklan
    teks = re.sub(
        r"ADVERTISEMENT",
        " ",
        teks,
        flags=re.IGNORECASE
    )


    teks = re.sub(
        r"SCROLL TO CONTINUE WITH CONTENT",
        " ",
        teks,
        flags=re.IGNORECASE
    )


    # menghapus spasi berlebihan
    teks = re.sub(
        r"\s+",
        " ",
        teks
    )


    return teks.strip()

## Fungsi Ekstraksi Isi Berita

ungsi ambil_isi_berita digunakan untuk mengunduh halaman web dari URL artikel yang diberikan dan menggunakan library trafilatura untuk mengekstrak teks utama dari berita tersebut. Setelah teks diambil, teks tersebut langsung dibersihkan menggunakan fungsi bersihkan_teks sebelumnya, dengan syarat teks harus memiliki panjang minimal 300 karakter.

In [8]:
def ambil_isi_berita(url):

    try:

        response = session.get(
            url,
            timeout=25
        )

        response.raise_for_status()


        isi = trafilatura.extract(
            response.text,
            include_comments=False,
            include_tables=False,
            output_format="txt",
            favor_precision=True,
            deduplicate=True
        )


        isi = bersihkan_teks(isi)


        # menghindari halaman kosong
        # atau isi yang terlalu pendek
        if isi and len(isi) >= 300:

            return isi


    except Exception as e:

        print(
            "Gagal:",
            url
        )

        print(
            "Error:",
            e
        )


    return None

## Fungsi Scraping Konten Berdasarkan Kategori

Kode ini mendefinisikan fungsi scraping_kategori yang bertugas untuk melakukan perulangan (looping) pada daftar URL, mengekstrak teks beritanya satu per satu, dan menyimpan hasilnya beserta label kategorinya ke dalam bentuk list. Fungsi ini dilengkapi dengan delay secara acak (1 hingga 2 detik) agar proses scraping tidak terlalu membebani server target

In [9]:
def scraping_kategori(
    links,
    label,
    target=100
):

    hasil = []


    for url in tqdm(
        links,
        desc=f"Scraping {label}"
    ):


        if len(hasil) >= target:
            break


        isi = ambil_isi_berita(url)


        if isi is not None:

            hasil.append(
                {
                    "isi_berita": isi,
                    "label": label
                }
            )


        # delay supaya tidak terlalu cepat
        time.sleep(
            random.uniform(1, 2)
        )


    print(
        f"{label} berhasil:",
        len(hasil)
    )


    return hasil

## Eksekusi Scraping Konten Sport

Kode ini menjalankan fungsi scraping_kategori untuk tautan-tautan dari kategori "sport", dengan target mengambil isi teks utuh dari 100 artikel

In [10]:
data_sport = scraping_kategori(
    link_sport,
    "sport",
    target=100
)

Scraping sport:   0%|          | 0/157 [00:00<?, ?it/s]

Gagal: https://sport.detik.com/moto-gp/d-8654776/motogp-san-marino-2026-jaga-puncak-klasemen-bukan-prioritas-jorge-martin
Error: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


Scraping sport:   1%|          | 1/157 [00:02<05:26,  2.09s/it]

Gagal: https://sport.detik.com/raket/d-8654666/ganda-campuran-indonesia-kembali-rombak-pemain
Error: HTTPSConnectionPool(host='sport.detik.com', port=443): Max retries exceeded with url: /raket/d-8654666/ganda-campuran-indonesia-kembali-rombak-pemain (Caused by ConnectTimeoutError(<HTTPSConnection(host='sport.detik.com', port=443) at 0x1e6d7200a10>, 'Connection to sport.detik.com timed out. (connect timeout=25)'))


Scraping sport:   1%|▏         | 2/157 [00:45<1:07:46, 26.24s/it]

...

Gagal: https://sport.detik.com/sportstyle/d-8653076/ngedadak-padel-ketika-padel-jadi-ajang-reuni
Error: HTTPSConnectionPool(host='sport.detik.com', port=443): Max retries exceeded with url: /sportstyle/d-8653076/ngedadak-padel-ketika-padel-jadi-ajang-reuni (Caused by NameResolutionError("HTTPSConnection(host='sport.detik.com', port=443): Failed to resolve 'sport.detik.com' ([Errno 11001] getaddrinfo failed)"))


Scraping sport:  10%|█         | 16/157 [21:43<2:15:50, 57.81s/it]

Gagal: https://sport.detik.com/sportstyle/d-8653072/menjaga-gaya-hidup-sehat-lewat-hyrox
Error: HTTPSConnectionPool(host='sport.detik.com', port=443): Max retries exceeded with url: /sportstyle/d-8653072/menjaga-gaya-hidup-sehat-lewat-hyrox (Caused by NameResolutionError("HTTPSConnection(host='sport.detik.com', port=443): Failed to resolve 'sport.detik.com' ([Errno 11001] getaddrinfo failed)"))


Scraping sport:  76%|███████▋  | 120/157 [26:49<08:16, 13.41s/it] 

sport berhasil: 100


## Eksekusi Scraping Konten Finance

Kode ini menjalankan fungsi scraping_kategori untuk tautan-tautan dari kategori "finance", dengan target mengambil isi teks utuh dari 100 artikel

In [11]:
data_finance = scraping_kategori(
    link_finance,
    "finance",
    target=100
)

Scraping finance:  68%|██████▊   | 104/154 [04:20<02:05,  2.51s/it]

finance berhasil: 100


## Penggabungan Data

Kode ini menggabungkan hasil scraping artikel kategori sport dan kategori finance menjadi satu kumpulan data tunggal bernama data_semua.

In [12]:
data_semua = (
    data_sport
    +
    data_finance
)

## Pembuatan DataFrame

Data hasil penggabungan yang masih berupa list diubah ke dalam bentuk format tabel menggunakan Pandas DataFrame (df = pd.DataFrame(data_semua)) agar lebih mudah dikelola dan dianalisis.

In [13]:
df = pd.DataFrame(
    data_semua
)

## Penambahan Kolom ID

Kode ini menyisipkan kolom baru bernama "id" pada posisi indeks ke-0 (kolom pertama) pada tabel DataFrame. Kolom ID ini diisi dengan angka berurutan mulai dari 1 hingga jumlah baris data.

In [14]:
df.insert(
    0,
    "id",
    range(
        1,
        len(df) + 1
    )
)

## Penataan Urutan Kolom

Kode ini mengatur ulang susunan kolom pada tabel DataFrame agar secara berurutan menampilkan kolom "id", lalu "isi_berita", dan terakhir "label".

In [15]:
df = df[
    [
        "id",
        "isi_berita",
        "label"
    ]
]

## Menampilkan Data

Menjalankan variabel df secara mandiri agar antarmuka Jupyter Notebook menampilkan pratinjau tabel hasil akhir data scraping.

In [16]:
df

,id,isi_berita,label
0,1,Men's World Tennis Championship 2026 sudah mem...,sport
1,2,Barra Ghaisan Zeinatma menunjukkan perkembanga...,sport
2,3,Marco Bezzecchi tak sabar menghadapi MotoGP Sa...,sport
3,4,Marc Marquez masih harus beradaptasi dengan ko...,sport
4,5,Pebalap Mercedes GP Kimi Antonelli tampil luar...,sport
...,...,...,...
195,196,Jakarta - Bandara ditutup akibat abu vulkanik ...,finance
196,197,Penutupan operasional tujuh bandara terdampak ...,finance
197,198,Otoritas Jasa Keuangan (OJK) mencatat 58 dana ...,finance
198,199,AirNav Indonesia kembali memperbarui informasi...,finance


## Pengecekan Total Data

Kode ini menggunakan fungsi len(df) dan mencetaknya untuk memverifikasi serta menampilkan total baris (jumlah artikel) yang ada di dalam DataFrame secara keseluruhan

In [17]:
print(
    "Total data:",
    len(df)
)

Total data: 200


## Pengecekan Proporsi Label

Kode ini menghitung dan menampilkan jumlah data per kategori menggunakan fungsi value_counts() pada kolom "label", untuk memastikan keseimbangan jumlah data (contohnya 100 untuk sport dan 100 untuk finance).

In [18]:
print(
    df["label"].value_counts()
)

label
sport      100
finance    100
Name: count, dtype: int64


## Penyimpanan Data ke File Excel

Kode ini menyimpan (mengekspor) tabel DataFrame Pandas yang sudah final ke dalam sebuah file Excel dengan nama dataset_detik_200_berita.xlsx. Indeks bawaan Pandas tidak disertakan ke dalam file Excel tersebut.

In [19]:
nama_file = "dataset_detik_200_berita.xlsx"


df.to_excel(
    nama_file,
    index=False
)


print(
    "File berhasil disimpan:",
    nama_file
)

File berhasil disimpan: dataset_detik_200_berita.xlsx


## Pengecekan Lokasi File

Menggunakan modul os bawaan Python, kode ini melacak dan mencetak path absolut (lokasi spesifik di dalam memori/komputer) tempat file Excel hasil scraping tersebut baru saja disimpan.

In [20]:
import os

print(os.path.abspath("dataset_detik_200_berita.xlsx"))

C:\Users\safit\Documents\PPW\dataset_detik_200_berita.xlsx
